# 🎯 Foto-KI neu trainieren (Schuss-Challenge)

Dieses Notebook trainiert das **YOLO-Detektionsmodell** (Klassen `discipline` + `score`),
das die App nutzt, um Disziplin und Ergebnis von einem Trefferanlagen-Monitor zu lesen –
und exportiert es **fertig zum Drop-in** ins Repo.

**So benutzt du es (in [Google Colab](https://colab.research.google.com/) öffnen):**
1. Oben **Laufzeit → Laufzeittyp ändern → GPU (T4)** wählen.
2. Zellen der Reihe nach mit ▶ ausführen.
3. In **Schritt 2** deine Datenquelle wählen (Synthetik zum Test **ohne Fotos**, oder echte Fotos).
4. Am Ende lädt sich `vision_model_dropin.zip` herunter → ins Repo legen (Schritt 6).

> Das Einzige, was *du* für hohe Real-Genauigkeit liefern musst, sind **echte Fotos** (Schritt 2,
> Variante B). Alles andere ist automatisch.


## Schritt 0 – GPU prüfen & Ultralytics installieren


In [ ]:
!nvidia-smi -L  # zeigt die zugewiesene GPU (falls Fehler: Laufzeittyp -> GPU)
%pip -q install ultralytics pillow
import ultralytics; ultralytics.checks()

## Schritt 1 – Konfiguration

Muss zur App passen (`image-compare-brain.js` → `VISION_MODEL`). Standard ist korrekt.


In [ ]:
IMGSZ   = 640                 # == VISION_MODEL.inputSize
EPOCHS  = 100                 # mehr = besser/langsamer; 100 ist ein guter Start
BATCH   = 16
MODEL   = 'yolo11n.pt'        # klein & schnell; Alternative: 'yolov8n.pt'
CLASSES = ['discipline', 'score']   # Reihenfolge == metadata.yaml 'names'
print('Konfig:', IMGSZ, EPOCHS, BATCH, MODEL, CLASSES)

## Schritt 2 – Datensatz wählen

Führe **genau eine** der drei Varianten aus. Sie setzen jeweils `DATA` (Pfad zur `data.yaml`).

### Variante A — Synthetik (0 Fotos, sofort lauffähig)
Erzeugt automatisch gelabelte Monitor-Bilder. Ideal, um die ganze Pipeline **ohne ein einziges
eigenes Foto** zu testen. Für maximale Real-Genauigkeit später echte Fotos beimischen (Variante B).


In [ ]:
!git clone -q https://github.com/kr511/schuss-challenge
!python schuss-challenge/training/generate_synthetic_monitor.py \
    --out /content/datasets/monitor --count 1000 --imgsz 640
DATA = '/content/datasets/monitor/data.yaml'
print('DATA =', DATA)

### Variante B — Echte Fotos via Roboflow (beste Real-Genauigkeit)
1. Fotos auf [roboflow.com](https://roboflow.com) hochladen, je **discipline** und **score** als Box labeln
   (genau diese Klassennamen!), als **YOLOv11/YOLOv8** exportieren.
2. Roboflow zeigt dir ein `pip install roboflow … dataset.download(...)`-Snippet — hier einfügen.
3. `DATA` auf die heruntergeladene `data.yaml` setzen.

💡 **Tipp:** Synthetik + echte Fotos kombinieren = wenig Label-Arbeit, robustes Modell.
Dazu die Bilder/Labels beider Sätze in einen gemeinsamen Ordner kopieren.


In [ ]:
# %pip -q install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key='DEIN_KEY')
# dataset = rf.workspace('...').project('...').version(1).download('yolov11')
# DATA = dataset.location + '/data.yaml'
# print('DATA =', DATA)

### Variante C — Eigenes YOLO-ZIP hochladen
Bereits gelabelte Daten (Ordner `images/`, `labels/`, `data.yaml`) als ZIP.


In [ ]:
# from google.colab import files; up = files.upload()   # dein .zip
# import zipfile, os
# name = list(up)[0]
# zipfile.ZipFile(name).extractall('/content/mydata')
# DATA = '/content/mydata/data.yaml'   # ggf. Pfad anpassen
# print('DATA =', DATA)

## Schritt 3 – Training


In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(data=DATA, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
                      patience=25, name='monitor')
print('Bestes Gewicht:', results.save_dir)

## Schritt 4 – Validieren
`mAP50` nahe **1.0** = sehr gut. Bei Synthetik üblich; bei echten Fotos ist >0.8 schon stark.


In [ ]:
best = f'{results.save_dir}/weights/best.pt'
metrics = YOLO(best).val(data=DATA, imgsz=IMGSZ)
print('mAP50:', round(float(metrics.box.map50), 4), '| mAP50-95:', round(float(metrics.box.map), 4))

## Schritt 5 – Export nach TensorFlow.js + Drop-in-Paket


In [ ]:
import glob, os, shutil, yaml
YOLO(best).export(format='tfjs', imgsz=IMGSZ)
web = sorted(glob.glob(os.path.join(os.path.dirname(best), '*_web_model')))[-1]
print('Export-Ordner:', web)
print('Dateien:', sorted(os.listdir(web)))

# metadata.yaml mit den Klassennamen beilegen (für den Repo-Check).
with open(os.path.join(web, 'metadata.yaml'), 'w') as f:
    yaml.safe_dump({'names': {i: n for i, n in enumerate(CLASSES)}}, f, sort_keys=False)

shutil.make_archive('/content/vision_model_dropin', 'zip', web)
from google.colab import files
files.download('/content/vision_model_dropin.zip')
print('\nFertig! vision_model_dropin.zip wurde heruntergeladen.')

## Schritt 6 – Ins Repo übernehmen (Drop-in)

`vision_model_dropin.zip` enthält `model.json`, `group1-shard*of*.bin`, `metadata.yaml`.

**Am einfachsten:** ZIP an Claude Code geben mit *„übernimm dieses Modell als Drop-in"* —
Claude legt die Dateien richtig ab, entfernt veraltete Shards, zählt `VISION_MODEL.version`
hoch und hebt die Cache-Version an.

**Manuell:**
1. Inhalt entpacken und `model.json`, alle `group1-shard*of*.bin`, `metadata.yaml`
   ins **Repo-Root** kopieren (alte Shards mit abweichender Anzahl löschen!).
2. In `image-compare-brain.js` → `VISION_MODEL.version` hochzählen.
3. `npm run check:vision-model` ausführen (prüft Shards + Klassen).
4. Cache-Busting in `index.html`/`sw.js` anheben, committen, pushen.

Details: `docs/vision-model-upgrade.md`.
